# API Support Debug Lab — Colab walkthrough

A reproducible developer-support debugging lab written in Rust. This notebook walks the crate end-to-end on a fresh Colab VM in roughly **two minutes** after the Rust toolchain installs.

**What you'll see.** Eight diagnostic rules. Thirteen bundled positive fixtures plus ten negatives. Three real-API webhook envelopes (Stripe v1, Slack v0, GitHub HMAC). A Brier-calibrated confidence model with a regression canary. Ninety-plus tests across eleven groups, all green.

**What this is not.** A web service. A live API. A learned classifier. The lab is local, offline, rule-based, and honest about every claim it makes.

**Honest note.** This notebook expects the repo to be reachable at `github.com/infinityabundance/api-support-debug-lab`. Until the maintainer pushes the repo to GitHub, the `git clone` step in cell 3 will fail; everything else assumes that step succeeded.

In [ ]:
# Install a minimal Rust toolchain. ~90 s on Colab the first time.
# `--profile minimal` skips docs and the rust-src component we don't need here.
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable
import os
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!rustc --version && cargo --version

In [ ]:
# Clone the repo. Requires the repo to be reachable at this URL.
!git clone --depth 1 https://github.com/infinityabundance/api-support-debug-lab.git
%cd api-support-debug-lab

In [ ]:
# Release build so the demo runs fast (~30 s wall-clock on Colab).
# `tail -5` keeps the output cell tight; the build itself is uneventful.
!cargo build --release 2>&1 | tail -5

In [ ]:
# Thirteen bundled positive fixtures, one per directory under fixtures/cases/.
# Negatives and the _calibration set are hidden from this listing by convention.
!./target/release/api-debug-lab list-cases

In [ ]:
# The money shot: the human report a support engineer would paste into a ticket.
# Notice the EVIDENCE block (three structural signals, not just '401'), the
# REPRODUCTION block (a deterministic curl command), the NEXT STEPS block,
# and the ESCALATION NOTE block that names the divergence space.
!./target/release/api-debug-lab diagnose auth_missing

In [ ]:
# Two rules fire on this case: webhook_signature_mismatch (HMAC differs) AND
# webhook_timestamp_stale (drift outside tolerance). The orchestrator ranks
# them by confidence — HMAC mismatch is dispositive evidence, timestamp
# drift has benign causes — and surfaces the loser as ALSO CONSIDERED.
!./target/release/api-debug-lab diagnose webhook_signature_invalid_stale

In [ ]:
# Slack-style envelope: `X-Slack-Signature: v0=<hex>` over the signing input
# `"v0:{timestamp}:{body}"`. The rule recomputes the HMAC against the
# bundled secret and reports the mismatch. The two other supported envelopes
# (Stripe v1, GitHub HMAC) are exercised by their own fixtures.
!./target/release/api-debug-lab diagnose webhook_slack_v0

In [ ]:
# Sweep every case.json under fixtures/cases/ (positives, negatives, and
# the _calibration edge cases). Exit code is non-zero if any case is
# unclassified — useful as a regression check when adding rules.
!./target/release/api-debug-lab corpus fixtures/cases | tail -25

In [ ]:
# Full test suite: per-rule unit tests, schema validation, CLI integration
# via assert_cmd, snapshot tests via insta, property-based tests via proptest,
# calibration (aggregate Brier, per-rule Brier, ECE), oracle differential
# tests against externally-computed HMAC references, and a per-rule latency
# budget. ~5 s on Colab.
!cargo test --tests 2>&1 | grep 'test result'

In [ ]:
# The calibration test enforces five empirical properties over the 36-case
# labelled corpus: 100% primary-classification accuracy, aggregate Brier
# ≤ 0.05, per-rule Brier ≤ 0.08, ECE ≤ 0.05, and zero confidence on
# unclassified cases. The rubric is documented in docs/confidence_model.md.
!cargo test --test calibration 2>&1 | tail -15

## Optional: deeper rigour cells

These are reproducible from the same VM but each adds 3–5 minutes (mostly install time for the tool). Copy-paste into a fresh code cell to run.

**Mutation testing** — 91 % kill rate over 169 viable mutants in `src/rules.rs`:

```python
!cargo install --locked cargo-mutants
!cargo mutants --in-place --file src/rules.rs --no-shuffle --timeout-multiplier=2
```

**Code coverage** — 92 % regions, 93 % lines:

```python
!cargo install --locked cargo-llvm-cov
!rustup component add llvm-tools-preview
!cargo llvm-cov --summary-only
```

**Microsecond per-case benchmarks**:

```python
!cargo bench --bench diagnose -- --quick 2>&1 | grep 'time:'
```

**Calibration regression canary** — simulates a deliberately-miscalibrated rule and asserts the production Brier check would catch it; proves the calibration framework is load-bearing rather than ceremonial:

```python
!cargo test --features calibration_canary --test calibration_regression
```

Measured numbers live in `docs/mutation_report.md`, `docs/coverage.md`, and `docs/benchmarks.md`. The confidence rubric is in `docs/confidence_model.md`. Three architecture decision records explain the major design choices under `docs/adr/`.